# Exploring the construction permits for the Tableau test (layer 23)

The question: how many **business days** does it take to approve a new Public Space Construction Permit in fiscal year 2026 (1 October 2025 – 30 September 2026), overall, by permit type, and by another useful split?

There is no data dictionary, so before building anything I checked:

1. **Load:** the layer and its key columns
2. **Which permits:** new, issued in FY26, applied on or after 1 October 2025, one row per permit
3. **What "approve" means:** which date marks the approval
4. **Missing permits:** whether the layer has already dropped some FY26 permits
5. **Permit types:** how to group them into categories
6. **Other splits:** which segments show real differences

Each section ends with the decision it led to.

*This notebook uses the Python test's environment (`python_test/.venv`, kernel "CS12 Python Test (3.11)"), since it needs the same packages. The same layer was explored for the GIS test (renewal permits); here the focus is new permits.*

In [1]:
import pandas as pd
import requests

pd.set_option("display.width", 160)

PERMITS_23 = "https://maps2.dcgis.dc.gov/dcgis/rest/services/DCGIS_APPS/DDOT_TOPS/MapServer/23"
TIMEZONE = "America/New_York"
FY26_START = pd.Timestamp("2025-10-01", tz=TIMEZONE)
FY27_START = pd.Timestamp("2026-10-01", tz=TIMEZONE)

## 1. Load the data

The whole layer, page by page (the service hands out at most 2,000 rows per request).

In [2]:
# The layer's own description: column names, types and code lists
layer_info = requests.get(PERMITS_23, params={"f": "json"}, timeout=60).json()

# Ask for the rows page by page until the service says there are no more
rows, offset = [], 0
while True:
    params = {
        "where": "1=1",
        "outFields": "*",
        "returnGeometry": "false",
        "orderByFields": "OBJECTID",  # same order every time, so pages don't overlap
        "resultOffset": offset,
        "resultRecordCount": 2000,
        "f": "json",
    }
    page = requests.get(f"{PERMITS_23}/query", params=params, timeout=120).json()
    if "error" in page:               # a bad query still comes back as "OK"
        raise RuntimeError(page["error"])
    rows += [feature["attributes"] for feature in page["features"]]
    if not page.get("exceededTransferLimit"):
        break
    offset += len(page["features"])
permits = pd.DataFrame(rows)

# Dates arrive as milliseconds since 1970 in UTC, so convert them to DC time
for field in layer_info["fields"]:
    if field["type"] == "esriFieldTypeDate":
        permits[field["name"]] = pd.to_datetime(permits[field["name"]], unit="ms", utc=True).dt.tz_convert(TIMEZONE)

print(f"{len(permits):,} rows, {permits['TrackingNumber'].nunique():,} permits, {permits.shape[1]} columns")
print("Date columns:", [f["name"] for f in layer_info["fields"] if f["type"] == "esriFieldTypeDate"])

55,332 rows, 10,142 permits, 100 columns
Date columns: ['IntakeDate', 'ApplicationDate', 'ReadyForReviewDate', 'AssignedDate', 'LastApprovalDate', 'IssueDate', 'EffectiveDate', 'ExpirationDate', 'PermitFeesPaymentDate', 'InspectionPaymentDate', 'MeterPaymentDate', 'LastUpdateDate', 'PCReviewStatusDate', 'LastResubmissionDate', 'LastResubmissionRequestDate', 'CellInstallationDate']


## 2. Which permits

The brief: application date on or after 1 October 2025, and only **new** permits that have been **issued in FY26**. From the GIS exploration, `Renewal = 1` means New, and a permit counts as issued when it has an issue date (it keeps it when its status later moves to "Permit Expired").

In [3]:
# Apply the brief's filters one at a time and count what's left
steps = {
    "All rows": pd.Series(True, index=permits.index),
    "New (Renewal = 1)": permits["Renewal"] == 1,
    "Applied on/after 1 Oct 2025": permits["ApplicationDate"] >= FY26_START,
    "Issued (has an issue date)": permits["IssueDate"].notna(),
    "Issued in FY26": (permits["IssueDate"] >= FY26_START) & (permits["IssueDate"] < FY27_START),
}
keep = pd.Series(True, index=permits.index)
counts = []
for step, rule in steps.items():
    keep &= rule
    counts.append({"filter": step, "rows": keep.sum(), "permits": permits.loc[keep, "TrackingNumber"].nunique()})
print(pd.DataFrame(counts).to_string(index=False))

fy26 = permits[keep]
print()
print(fy26["StatusDescription"].value_counts())

                     filter  rows  permits
                   All rows 55332    10142
          New (Renewal = 1) 27157     5536
Applied on/after 1 Oct 2025 22838     4981
 Issued (has an issue date) 13218     3061
             Issued in FY26 12155     2770

StatusDescription
Issued            10420
Permit Expired     1723
Suspended            11
Revoked               1
Name: count, dtype: int64


In [4]:
# A permit has one row per location. Are the permit-level columns the same on all its rows?
permit_columns = ["ApplicationDate", "IssueDate", "LastApprovalDate", "StatusDescription", "TypeDetailNames",
                  "IsExcavation", "IsPaving", "IsOccupancy", "ReviewProcess", "PermitTypeDescription",
                  "DaysToApprove", "DaysToIssue", "PermitteeName", "TotalFee"]
differs = {column: (fy26.groupby("TrackingNumber")[column].nunique(dropna=False) > 1).sum() for column in permit_columns}
print("Permits where a column differs between its rows:", differs)
print()
print("Rows per permit:", fy26.groupby("TrackingNumber").size().describe().round(1).to_dict())

# Permits applied in FY26 but issued after it ended
late = permits[(permits["Renewal"] == 1) & (permits["ApplicationDate"] >= FY26_START) & (permits["IssueDate"] >= FY27_START)]
print(f"Applied in FY26 but issued after 30 Sep 2026: {late['TrackingNumber'].nunique()} permits (excluded)")

Permits where a column differs between its rows: {'ApplicationDate': np.int64(0), 'IssueDate': np.int64(0), 'LastApprovalDate': np.int64(0), 'StatusDescription': np.int64(0), 'TypeDetailNames': np.int64(0), 'IsExcavation': np.int64(0), 'IsPaving': np.int64(0), 'IsOccupancy': np.int64(0), 'ReviewProcess': np.int64(0), 'PermitTypeDescription': np.int64(0), 'DaysToApprove': np.int64(0), 'DaysToIssue': np.int64(0), 'PermitteeName': np.int64(0), 'TotalFee': np.int64(0)}

Rows per permit: {'count': 2770.0, 'mean': 4.4, 'std': 7.8, 'min': 1.0, '25%': 1.0, '50%': 1.0, '75%': 2.0, 'max': 75.0}
Applied in FY26 but issued after 30 Sep 2026: 291 permits (excluded)


**What this shows**

- About 2,770 new permits applied for on or after 1 October 2025 were issued in FY26. They have about 12,000 rows, because one permit can list many locations (up to 75).
- Every permit-level column (dates, status, type, fees, review process) is the same on all of a permit's rows.
- Most of these permits still have the status "Issued"; the rest moved on to "Permit Expired", "Suspended" or "Revoked" after being issued.
- Some permits applied for in FY26 were only issued after 30 September 2026. They're outside FY26 and left out.

**Decision:** the analysis uses **one row per permit** (`TrackingNumber`). Counting rows would weight a permit with 75 locations 75 times.